In [1]:
import os
base_path = os.path.dirname(os.path.abspath(""))
base_path = base_path + "/headers/"
print(base_path)
header_path = base_path + 'header_data_treatment.ipynb'
%run $header_path

C:\Users\raphael.wargon\Documents\GitHub\sorting_univ\script/headers/
initalizing spark nlp session w gpu


In [2]:
df_authors = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'authors.parquet')
              .withColumn('max_updated_date', func.max(func.col('updated_date')).over(Window.partitionBy('id')))
              .filter(func.col('updated_date')== func.col('max_updated_date'))
              .withColumn('split_name', func.split(func.col('display_name'), ' '))
             .withColumn('first_name', func.element_at(func.col('split_name'), 1))
              .withColumn('last_name', func.concat_ws(' ', func.array_except(func.col('split_name'), func.array(func.col('first_name')))))
              .select(func.regexp_replace(func.col('id'), 'https://openalex.org/', '').alias('author_id'),
                      'orcid', 'display_name_alternatives', 'affiliations')
             )


In [3]:
df_inst = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'institutions.parquet')
           .withColumn('inst_id', func.regexp_replace(func.col('id'), 'https://openalex.org/', ''))
          )

In [4]:
au_level_char= (spark.read.parquet('file:\\' + int_save_path + 'au_level_char.parquet')
                .join(df_authors, on = 'author_id', how = 'inner')
               )
#au_level_char.cache()
#au_level_char.show()

In [5]:
au_level_char_w_inst = (au_level_char
                        .withColumn('affiliations', func.explode(func.col('affiliations.institution')))
                        .withColumn('inst_id', func.regexp_replace(func.col('affiliations.id'), 'https://openalex.org/', '') )
                        .join(df_inst.select('inst_id', 'geo.city', 'associated_institutions', func.concat_ws(' ', func.col('display_name_acronyms'))), on = 'inst_id', how = 'left')
                        .groupBy(*[col for col in au_level_char.columns if col !='affiliations'])
                        .agg( func.collect_set("affiliations").alias('affiliations'),
                             func.collect_set('city').alias('city'),
                             func.first('associated_institutions').alias('associated_institutions'))
                       )
au_level_char_w_inst.cache()
au_level_char_w_inst.show()

+-----------+----------+---------+---------------+------------+-------------+--------------------+------+-----+---------+---------+-----------------+------+--------------------+-------------------------+--------------------+--------------------+-----------------------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|         author_name|domain|field| subfield|    topic|author_first_name|gender|               orcid|display_name_alternatives|        affiliations|                city|associated_institutions|
+-----------+----------+---------+---------------+------------+-------------+--------------------+------+-----+---------+---------+-----------------+------+--------------------+-------------------------+--------------------+--------------------+-----------------------+
|A5000008405|      2012|     2017|              0|           2|           GB|      Hideyuki Maeda|     3|   17|     1710|     1710|         Hideyuki|     M|                NULL|     [Hideyuk

In [ ]:
au_level_char.printSchema()

In [6]:
anr_partenaires_2005 = (spark.read.csv('file:///'+ save_path + 'anr-dgds-2005-2009-projets-finances-20251105-partenaires.csv', header = True, sep = ';')
                                           .withColumn('Projet.Partenaire.Code_RNSR', func.lit(''))

                       )

anr_partenaires_2010 = (spark.read.csv('file:///'+ save_path + 'anr-dgds-depuis-2010-projets-finances-20260402-partenaires.csv', header = True, sep = ';')
                        )

In [7]:
anr_partenaires_2010

DataFrame[Projet.Code_Decision: string, Projet.Acronyme: string, Projet.Partenaire.Code_Decision: string, Projet.Partenaire.Est_coordinateur: string, Projet.Partenaire.Nom_organisme: string, Projet.Partenaire.Categorie_organisme: string, Projet.Partenaire.Code_RNSR: string, Projet.Partenaire.Responsable_scientifique.Nom: string, Projet.Partenaire.Responsable_scientifique.Prenom: string, Projet.Partenaire.Responsable_scientifique.ORCID: string, Projet.Partenaire.Adresse.Ville: string, Projet.Partenaire.Adresse.Region: string, Projet.Partenaire.Adresse.Pays: string, Projet.Partenaire.Aide_allouee.ANR: string, Projet.Partenaire.Financement.Source: string]

In [8]:

anr_partenaires = (anr_partenaires_2005
                        .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2005.columns])
                   .select(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2010.columns])
                   .unionAll(anr_partenaires_2010
                             .toDF(*[col.replace('Projet.', '').replace('Partenaire.','P_')
                                         .replace('.', '__') for col in anr_partenaires_2010.columns])
                            )
                  )

In [9]:
anr_partenaires.cache()
anr_partenaires.show()

+----------------+------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------------------------+----------------+--------------------+---------------+-------------------+---------------------+
|   Code_Decision|    Acronyme|    P_Code_Decision|P_Est_coordinateur|     P_Nom_organisme|P_Categorie_organisme|P_Code_RNSR|P_Responsable_scientifique__Nom|P_Responsable_scientifique__Prenom|P_Responsable_scientifique__ORCID|P_Adresse__Ville|   P_Adresse__Region|P_Adresse__Pays|P_Aide_allouee__ANR|P_Financement__Source|
+----------------+------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------------------------+----------------+--------------------+---------------+-------------------+---------------------+
|ANR-09-VPTT-0015|ATAC-CONCEPT|

In [10]:
anr_partenaires.count()

102430

In [11]:
anr_partenaires.filter(func.col('P_Adresse__Pays')=="France").count()

83779

In [12]:
merged_orcid = (anr_partenaires
                .filter(func.col('P_Responsable_scientifique__ORCID').isNotNull())
                .join(au_level_char
                      .select('author_id', func.regexp_replace(func.col('orcid'), 'https://orcid.org/', '' ).alias('P_Responsable_scientifique__ORCID'))
                      , on = 'P_Responsable_scientifique__ORCID', how = 'inner')
               )
merged_orcid.count()

37768

In [ ]:
accented = r"àâäáçèéêëîïìíôöòóùûüúÿ"
plain     = r"aaaaceeeeiiiioooouuuuy"

first_merge = (anr_partenaires
               .join(merged_orcid.select('P_Responsable_scientifique__ORCID'), on ='P_Responsable_scientifique__ORCID', how = 'left_anti')
               .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.col('P_Responsable_scientifique__Prenom'),
                                                                             func.col('P_Responsable_scientifique__Nom'))) )
               .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
               .withColumn('affiliation_normalized',  func.lower(func.regexp_replace(func.col('P_Nom_organisme'),  r'[^A-z\s\-]*', '')) )
               .join(au_level_char_w_inst
                     .select('author_id',
                             func.lower(func.regexp_replace(func.col('author_name'), r'(?<=\s)[A-Z]\.*\s', '')).alias('normalized_name'), 
                             func.transform(func.col('affiliations.display_name'), lambda x :
                                            func.lower(func.regexp_replace(func.translate(x,  accented, plain), r'[^A-z\s\-]*', '')) ).alias('affiliations'),
                             func.transform(func.col('associated_institutions.display_name'), lambda x :
                                            func.lower(func.regexp_replace(func.translate(x,  accented, plain), r'[^A-z\s\-]*', '')) ).alias('associated_institutions'),

                            'city'
                            )
                     .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))

                     , on = 'normalized_name', how = 'left')
               .distinct()
               .withColumn('n_match', func.size(func.collect_set('author_id').over(Window.partitionBy('normalized_name'))))
               .withColumn('common_af', func.array_max(func.transform(func.col('affiliations'), 
                                                       lambda x : ( (x.contains(func.col('affiliation_normalized')) | func.col('affiliation_normalized').contains(x)) ).cast('int')
                                                      )) )
               .withColumn('common_city', func.array_max(func.transform(func.col('city'), 
                                                       lambda x : ( ( func.lower(x).contains(func.lower(func.col('P_Adresse__Ville')) ) 
                                                                     | func.lower(func.col('P_Adresse__Ville')).contains( func.lower(x) )) ).cast('int')
                                                      )) )
               .fillna(0,['common_af','common_city'])
               .withColumn('associated_inst', func.array_max(func.transform(func.col('associated_institutions'), 
                                                       lambda x : ( (x.contains(func.col('affiliation_normalized')) | func.col('affiliation_normalized').contains(x)) ).cast('int')
                                                      )) )

               .withColumn('matched', ( (func.col('n_match')==1) | (func.col('common_af')==1) | (func.col('common_city')==1 ) | (func.col('associated_inst') == 1 ) ).cast('int'))
               .withColumn('max_matched', func.max('matched').over(Window.partitionBy('normalized_name')))
              )
first_merge.cache()
first_merge.groupBy('matched').count().show()

In [17]:
first_merge.filter('n_match=2').filter('max_matched=0').filter('associated_inst=1').show(truncate = False)

+-----------------+---------------------------------+----------------+--------------+-------------------+------------------+--------------------------------------------------------------------------------------------------------+----------------------+-----------+-------------------------------+----------------------------------+----------------+--------------------------+---------------+-------------------+---------------------+------------------------------------------------------------------------------------------------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
first_merge.filter('n_match=2').filter('max_matched=0').show(truncate = False)

+-------------------+---------------------------------+----------------+------------+-------------------+------------------+------------------------------------------------------------------------------+------------------------+-----------+-------------------------------+----------------------------------+----------------+--------------------+---------------+-------------------+---------------------+--------------------------------------------------------------------------+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
first_merge.filter('n_match=2').filter('common_city=1').count()

4405

In [16]:
first_merge.filter('n_match>1').groupBy( (func.array_max(func.col('common_af'))==1).alias('common_af')).count().show(truncate = False)

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "array_max(common_af)" due to data type mismatch: Parameter 1 requires the "ARRAY" type, however "common_af" has the type "INT".;
'Aggregate [(array_max(common_af#6213) = 1)], [(array_max(common_af#6213) = 1) AS common_af#10175, count(1) AS count#10204L]
+- Filter (n_match#6114 > 1)
   +- Project [normalized_name#6006, P_Responsable_scientifique__ORCID#4129, Code_Decision#4121, Acronyme#4122, P_Code_Decision#4123, P_Est_coordinateur#4124, P_Nom_organisme#4125, P_Categorie_organisme#4126, P_Code_RNSR#4135, P_Responsable_scientifique__Nom#4127, P_Responsable_scientifique__Prenom#4128, P_Adresse__Ville#4130, P_Adresse__Region#4131, P_Adresse__Pays#4132, P_Aide_allouee__ANR#4133, P_Financement__Source#4134, affiliation_normalized#6023, author_id#6059, affiliations#6042, associated_institutions#6043, city#349, n_match#6114, common_af#6213, common_city#6214, ... 3 more fields]
      +- Project [normalized_name#6006, 

In [104]:
first_merge.groupBy('n_match').count().show()

+-------+-----+
|n_match|count|
+-------+-----+
|      1|54326|
|      3| 8028|
|      4| 3900|
|      8|  760|
|      7| 1169|
|      2|17294|
|      0|34367|
|     26|  156|
|      6| 1764|
|     17|  102|
|     71|  355|
|     46|   92|
|     34|  204|
|     14|  420|
|     15|  225|
|      5| 2490|
|     13|  286|
|    170|  170|
|     11|  352|
|     10|  510|
+-------+-----+
only showing top 20 rows



+--------------------+----------------+------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------------------------+----------------+--------------------+---------------+-------------------+---------------------+-----------+--------------------+-------+-------+
|     normalized_name|   Code_Decision|    Acronyme|    P_Code_Decision|P_Est_coordinateur|     P_Nom_organisme|P_Categorie_organisme|P_Code_RNSR|P_Responsable_scientifique__Nom|P_Responsable_scientifique__Prenom|P_Responsable_scientifique__ORCID|P_Adresse__Ville|   P_Adresse__Region|P_Adresse__Pays|P_Aide_allouee__ANR|P_Financement__Source|  author_id|         author_name|matched|n_match|
+--------------------+----------------+------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+-

In [141]:
first_merge.filter(func.col('P_Adresse__Pays')=="France").groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|   NULL|18178|
|      1|30258|
|      0|14734|
+-------+-----+



In [117]:
unmatched = (anr_partenaires
               .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.col('P_Responsable_scientifique__Prenom'),
                                                                             func.col('P_Responsable_scientifique__Nom'))) )
               .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
             .join(first_merge
                   .filter('matched=0')
                   .select('normalized_name'), on = 'normalized_name', how = 'inner')
            )

In [118]:
unmatched.distinct().count()

39421

In [119]:
second_merge = (unmatched
                .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.substring(func.col('P_Responsable_scientifique__Prenom'), 1, 1),
                                                                         func.col('P_Responsable_scientifique__Nom'))))
                .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
               .join(au_level_char
                     .select('author_id', 'author_name',
                             func.lower(func.regexp_replace(func.col('author_name'), r'(?<=\s)[A-Z]\.*\s', '')).alias('normalized_name') )
                     .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
                     .withColumn('normalized_name', func.regexp_replace(func.col('normalized_name'), r'\.', ""))
                     , on = 'normalized_name', how = 'left')
                .withColumn('matched', (func.col('author_id').isNotNull()).cast('int'))
                .distinct()

               )
second_merge.groupBy('matched').count().show()

+-------+-----+
|matched|count|
+-------+-----+
|      1|57572|
|      0|26585|
+-------+-----+



In [ ]:
(anr_partenaires
               .withColumn('normalized_name', func.lower(func.concat_ws(' ', func.col('Responsable_scientifique__Prenom'),
                                                                             func.col('Responsable_scientifique__Nom'))) )
               .withColumn('normalized_name', func.translate(func.col('normalized_name'), accented, plain))
 .filter(func.col('Responsable_scientifique__Nom' )=="STUDER").select('normalized_name')).show()

In [93]:
first_merge.filter('matched=0').show()

+--------------------+----------------+--------------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------------------------+----------------+--------------------+---------------+-------------------+---------------------+---------+-----------+-------+
|     normalized_name|   Code_Decision|            Acronyme|    P_Code_Decision|P_Est_coordinateur|     P_Nom_organisme|P_Categorie_organisme|P_Code_RNSR|P_Responsable_scientifique__Nom|P_Responsable_scientifique__Prenom|P_Responsable_scientifique__ORCID|P_Adresse__Ville|   P_Adresse__Region|P_Adresse__Pays|P_Aide_allouee__ANR|P_Financement__Source|author_id|author_name|matched|
+--------------------+----------------+--------------------+-------------------+------------------+--------------------+---------------------+-----------+-------------------------------+----------------------------------+---------------

In [94]:
au_level_char.filter(func.col("author_id")=="A5090731351").show()

+-----------+----------+---------+---------------+------------+-------------+-------------+------+-----+--------+-----+-----------------+------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|  author_name|domain|field|subfield|topic|author_first_name|gender|
+-----------+----------+---------+---------------+------------+-------------+-------------+------+-----+--------+-----+-----------------+------+
|A5090731351|      1992|     2020|             28|           2|        FR,GB|A. San Miguel|     3|   25|    2505| 2505|               A.|   INI|
+-----------+----------+---------+---------------+------------+-------------+-------------+------+-----+--------+-----+-----------------+------+



In [96]:
(au_level_char.filter(func.col("author_id")=="A5090731351")
                     .select('author_id', 'author_name',
                             func.lower(func.regexp_replace(func.col('author_name'), r'(?<=\s)[A-Z]\.*\s', '')).alias('normalized_name') )
                     .withColumn('normalized_name', func.trim(func.translate(func.col('normalized_name'), accented, plain)))

).show()

+-----------+-------------+---------------+
|  author_id|  author_name|normalized_name|
+-----------+-------------+---------------+
|A5090731351|A. San Miguel|  a. san miguel|
+-----------+-------------+---------------+



In [15]:
au_level_char.show()

+-----------+----------+---------+---------------+------------+-------------+--------------------+------+-----+---------+---------+-----------------+------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|         author_name|domain|field| subfield|    topic|author_first_name|gender|
+-----------+----------+---------+---------------+------------+-------------+--------------------+------+-----+---------+---------+-----------------+------+
|A5000001854|      2014|     2020|              0|           5|           GB|        Marco Polini|     3|   31|     3107|     3107|            Marco|     M|
|A5000002183|      2001|     2019|              0|          11|           GB|  Mashood A. Baderin|     2|   33|     3312|     3312|          Mashood|     F|
|A5000005233|      2002|     2020|             19|           0|           FR|         Gaël Thomas|     3|   17|     1705|     1705|             Gaël|     M|
|A5000005545|      1998|     1999|              0|        